# 영상의 군집은 토지피복인가?

스펙트럼 유사성과 실제 지표 종류를 구분

**할 일** · 같은 반사율 합성에서 군집 수 4·6 비교

위에서 아래로 실행. 코드를 바꾸기 전 결과를 예상하고, 실행 결과와 비교

## 실행 준비

CPU 런타임 사용. Earth Engine에 등록한 Cloud 프로젝트와 사용 권한 필요. [설정 안내](https://github.com/steelpl/lecture/blob/main/CE776-climate-change-carbon-water-cycle/GEE_SETUP.md) 참고. 데이터 조회에는 본인 계정 인증 필요

In [ ]:
%pip -q install earthengine-api==1.7.52 geemap==0.39.0

In [ ]:
import ee
import geemap

PROJECT_ID = ""  # Earth Engine에 등록한 본인의 Cloud 프로젝트 ID
if not PROJECT_ID.strip():
    raise ValueError("PROJECT_ID에 본인의 등록된 Cloud 프로젝트 ID 입력")
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

In [ ]:
roi = ee.Geometry.Rectangle([126.80, 37.43, 127.15, 37.70], geodesic=False)

In [ ]:
def mask_s2(image):
    scl = image.select('SCL')
    clear = (scl.neq(1).And(scl.neq(3)).And(scl.neq(8))
             .And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11)))
    return (image.select(['B2', 'B3', 'B4', 'B8', 'B11'])
            .multiply(0.0001).updateMask(clear)
            .copyProperties(image, ['system:time_start']))
collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
              .filterBounds(roi).filterDate('2020-05-01', '2020-10-01')
              .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 60))
              .map(mask_s2))

image = collection.median().clip(roi)
bands = ['B2','B3','B4','B8','B11']
sample = image.select(bands).sample(region=roi, scale=60, numPixels=2000, seed=7, geometries=False)
cluster4 = image.cluster(ee.Clusterer.wekaKMeans(4).train(sample))
cluster6 = image.cluster(ee.Clusterer.wekaKMeans(6).train(sample))

In [ ]:
print('source scenes:', collection.size().getInfo())
print('training samples:', sample.size().getInfo())

In [ ]:
Map = geemap.Map()
Map.centerObject(roi, 10)
Map.addLayer(image, {'bands':['B4','B3','B2'],'min':0,'max':0.3}, 'RGB')
Map.addLayer(cluster4.randomVisualizer(), {}, '4 clusters')
Map.addLayer(cluster6.randomVisualizer(), {}, '6 clusters')
Map

RGB 영상에서 물·도시·산지를 찾아 군집과 비교하시오. 군집 0을 곧바로 물이라고 부를 수 없는 이유를 설명하시오. 군집 수 변경으로 경계와 분할이 어떻게 달라지는지 설명하시오.

**해석** · 군집 번호는 임의의 범주. 스케일을 맞추지 않은 밴드 값은 거리 계산에 서로 다른 영향 가능

## 자료와 해석 범위

- [Sentinel-2 SR](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED)

관측 연도는 고정 사례의 기간. 수업 연도와 무관. 코드는 자료 처리 방법을 보여 주며 지도·수치는 인증 후 조회 결과로 판단